# ExoBaby — Long-Tailed Distribution Validation

**Replicates Finding 1 from:** *"Characterizing the visual representation of objects from the child's view"* (Yang et al., 2026 · arXiv:2605.14990)

## What this notebook does
1. Reads exocentric frames from the `filtered` folder in Google Drive
2. Downloads the 129 CDI reference categories used by the paper
3. Runs **YOLOE** open-vocabulary detection + **CLIP** filtering (same thresholds as paper)
4. Fits a **power-law** to the object frequency distribution
5. Saves all outputs to `ExoBaby-results/` in Google Drive

---

## Step 0 — Install packages

In [ ]:
# Step 0: clone the repo, set working directory, install packages.
#
# Opening a notebook from GitHub in Colab only fetches the .ipynb file —
# the rest of the repo (detection/ package, requirements file) is NOT present.
# We clone it here once per session. On reconnect it skips the clone.
#
# TODO: replace YOUR_USERNAME with your actual GitHub username.
REPO_URL  = 'https://github.com/tanzim12911/ExoBaby-pipeline.git'
REPO_NAME = 'ExoBaby-pipeline'

import os, subprocess

if not os.path.isdir(f'/content/{REPO_NAME}'):
    subprocess.run(['git', 'clone', REPO_URL, f'/content/{REPO_NAME}'], check=True)
    print('Repo cloned.')
else:
    print('Repo already present, skipping clone.')

os.chdir(f'/content/{REPO_NAME}')
print('Working directory:', os.getcwd())

In [ ]:
# Install all required packages
# ultralytics >= 8.3 is required for YOLOE open-vocabulary detection
# open-clip-torch is used for CLIP image-text filtering
!pip install -q 'ultralytics>=8.3' open-clip-torch powerlaw scipy pandas matplotlib tqdm requests pillow numpy huggingface_hub
print('All packages installed')

import os
import csv
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
import torch
from tqdm.notebook import tqdm
from PIL import Image
import open_clip
from huggingface_hub import hf_hub_download

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 12})
print('Imports done')

---
## Step 1 — Mount Google Drive & configure paths

Set `FRAMES_DIR` to where your `filtered/` folder lives in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Edit these lines before running ──────────────────────────────
FRAMES_DIR  = '/content/drive/Othercomputers/My laptop/data/filtered'
OUTPUT_BASE = '/content/drive/MyDrive/ExoBaby-results'
RUN_TAG     = 'v1_10k'   # describe this run: v2_12k, v3_threshold-0.25 ...

# PRUNE_STALE = False  -> warn about stale CSV rows but leave CSV untouched.
#                         Use this by default and when Drive may not be fully synced.
# PRUNE_STALE = True   -> remove CSV rows for frames no longer on disk.
#                         Only set this when you have intentionally deleted frames.
PRUNE_STALE = False
# ──────────────────────────────────────────────────

import os
import detection.config as cfg
from run_detection import resolve_shared_dirs, resolve_analysis_dirs

shared_dirs   = resolve_shared_dirs(OUTPUT_BASE)
analysis_dirs = resolve_analysis_dirs(OUTPUT_BASE, RUN_TAG)

dirs = {
    'data':       shared_dirs['data'],
    'frame_data': shared_dirs['frame_data'],
    'results':    analysis_dirs['results'],
    'figures':    analysis_dirs['figures'],
}

detection_csv = os.path.join(shared_dirs['frame_data'], cfg.DETECTION_CSV_NAME)

print(f'Frames dir      : {FRAMES_DIR}')
print(f'Detection CSV   : {detection_csv}')
print(f'Analysis output : {OUTPUT_BASE}/analysis/{RUN_TAG}/')
print(f'Prune stale     : {PRUNE_STALE}')

if not os.path.isdir(FRAMES_DIR):
    print(f'\n  WARNING: FRAMES_DIR not found: {FRAMES_DIR}')
    print('  Edit FRAMES_DIR above to match your actual Drive path.')
else:
    print('\n  Frames folder found!')
    if os.path.exists(detection_csv):
        import pandas as pd
        existing = len(pd.read_csv(detection_csv))
        print(f'  Existing detection CSV: {existing:,} rows')

In [ ]:
from run_detection import collect_frames

frame_paths = collect_frames(FRAMES_DIR)
print(f'Found {len(frame_paths):,} frames')
if frame_paths:
    print(f'  First: {frame_paths[0]}')
    print(f'  Last : {frame_paths[-1]}')

---
## Step 2 — Download CDI reference data

Downloaded once, then cached in `ExoBaby-results/data/`.

In [ ]:
from detection.cdi_loader import load_cdi_data

included_categories, lemma_to_semantic = load_cdi_data(dirs['data'])

print(f'Valid129 categories : {len(included_categories)}')
print(f'Semantic mappings   : {len(lemma_to_semantic)}')
print(f'First 15            : {included_categories[:15]}')

---
## Step 3 — YOLOE detection + CLIP filtering

> **Resumable:** Re-run this cell after a disconnect — already-processed frames are skipped automatically.

Pipeline:
1. YOLOE-v8-L detects objects using all 129 CDI class names (conf ≥ 0.25)
2. Each detection crop is compared to its label via CLIP ViT-B/32
3. Only detections with cosine similarity ≥ 0.27 are kept

In [ ]:
import torch

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32 if device == 'cuda' else 4
print(f'Device     : {device}')
if device == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('  No GPU — go to Runtime > Change runtime type > T4 GPU, then re-run all cells.')

In [ ]:
from detection.detector import load_models, set_cdi_classes, run_detection, precompute_text_embeddings
import detection.config as cfg

yoloe_model, clip_model, clip_preprocess, clip_tokenizer = load_models(device)
set_cdi_classes(yoloe_model, included_categories)

# Pre-compute CLIP text embeddings once — reused across all batches
# Eliminates repeated text encoding which was the main speed bottleneck
text_embeds = precompute_text_embeddings(included_categories, clip_model, clip_tokenizer, device)


In [ ]:
import os

detection_csv = os.path.join(dirs["frame_data"], cfg.DETECTION_CSV_NAME)

run_detection(
    frame_paths=frame_paths,
    included_categories=included_categories,
    output_csv=detection_csv,
    yoloe_model=yoloe_model,
    clip_model=clip_model,
    clip_preprocess=clip_preprocess,
    clip_tokenizer=clip_tokenizer,
    text_embeds=text_embeds,
    device=device,
    batch_size=batch_size,
    prune_stale=PRUNE_STALE,
)
print(f"\nDetection CSV: {detection_csv}")


In [ ]:
import pandas as pd

df_det_all = pd.read_csv(detection_csv)
print(f'Total detections  : {len(df_det_all):,}')
print(f'Unique frames hit : {df_det_all["frame_path"].nunique():,} / {len(frame_paths):,}')
print(f'Unique categories : {df_det_all["class_name"].nunique()} / {len(included_categories)}')
print('\nTop 20 detected categories:')
print(df_det_all['class_name'].value_counts().head(20).to_string())

---
## Step 4 — Long-tailed distribution analysis

Direct adaptation of Notebook 01 from the BabyView paper repository.

In [ ]:
from detection.analysis import build_category_summary, fit_power_law, print_summary

intermediate_csv = os.path.join(dirs['results'], cfg.INTERMEDIATE_CSV_NAME)

df_cat = build_category_summary(
    detection_csv=detection_csv,
    included_categories=included_categories,
    lemma_to_semantic=lemma_to_semantic,
    total_frames=len(frame_paths),
    output_csv=intermediate_csv,
)

print(f'Categories with >=1 detection: {(df_cat["count_frames"] > 0).sum()} / {len(included_categories)}')
print('\nTop 10:')
print(df_cat[['category', 'count_detected', 'count_frames', 'proportion', 'cdi_semantic']].head(10).to_string(index=False))

In [ ]:
fit = fit_power_law(df_cat)

print(f'Power-law exponent alpha : {fit["alpha"]:.3f}')
print(f'xmin                     : {fit["xmin"]:.1f}')
print(f'R (PL vs lognormal)      : {fit["r_vs_lognormal"]:.3f}  p = {fit["p_vs_lognormal"]:.4f}')
preferred = 'Power-law preferred' if fit['r_vs_lognormal'] > 0 else 'Lognormal preferred'
print(f'Verdict                  : {preferred}')

---
## Step 5 — Figures

In [ ]:
from detection.visualizer import plot_rank_bar, plot_loglog, plot_per_domain

total_detected = len(df_det_all)

bar_path = plot_rank_bar(df_cat, fit, len(frame_paths), total_detected, dirs['figures'])
print(f'Saved: {bar_path}')

In [ ]:
log_path = plot_loglog(fit['nonzero_df'], fit['counts_arr'], fit['alpha'], dirs['figures'])
print(f'Saved: {log_path}')

In [ ]:
dom_path = plot_per_domain(df_cat, dirs['figures'])
print(f'Saved: {dom_path}')

---
## Step 6 — Results summary

In [ ]:
print_summary(df_cat, fit, len(frame_paths), total_detected)

---
## Step 7 — Frame-level OOI coverage

Proportion of frames that contain at least one Object of Interest after CLIP filtering.

In [ ]:
import os, json

frames_with_detections = df_det_all['frame_path'].nunique()
total_frames           = len(frame_paths)
proportion             = frames_with_detections / total_frames if total_frames else 0.0

print(f'Total frames processed       : {total_frames:,}')
print(f'Frames with object detections: {frames_with_detections:,}')
print(f'Proportion of frames with detections: {proportion:.2%}')

# Save to results/ so it persists after Colab output is cleared
ooi_stats = {
    'run_tag':                  RUN_TAG,
    'total_frames':             total_frames,
    'frames_with_detections':   frames_with_detections,
    'proportion':               round(proportion, 6),
    'proportion_pct':           f'{proportion:.2%}',
}

ooi_json = os.path.join(dirs['results'], 'ooi_coverage.json')
with open(ooi_json, 'w', encoding='utf-8') as f:
    json.dump(ooi_stats, f, indent=2)

print(f'\nSaved to: {ooi_json}')

---
## Step 8 — Provenance analysis

For each CDI semantic domain, find the top-5 categories by detection count,
then break down each category by source video.

**What to look for:** if one video contributes >70% of detections for a category,
that category’s frequency reflects that video’s setting — not a general pattern.
This is a dataset bias check.

> This section reads from the saved detection CSV and does not require GPU.\
> Re-run it independently any time to refresh after adding new videos.

In [ ]:
import os
import detection.config as cfg
from detection.analysis import build_provenance

provenance_csv = os.path.join(
    dirs["results"],
    cfg.PROVENANCE_CSV_NAME.format(n=cfg.TOP_N_PER_DOMAIN),
)

# Rebuild df_cat if this cell is run standalone (new Colab session)
if "df_cat" not in dir():
    from run_detection import collect_frames
    from detection.analysis import build_category_summary
    from detection.cdi_loader import load_cdi_data
    _frame_paths = collect_frames(FRAMES_DIR)
    _included, _lemma_map = load_cdi_data(dirs["data"])
    df_cat = build_category_summary(
        detection_csv=detection_csv,
        included_categories=_included,
        lemma_to_semantic=_lemma_map,
        total_frames=len(_frame_paths),
        output_csv=os.path.join(dirs["results"], cfg.INTERMEDIATE_CSV_NAME),
    )

df_prov = build_provenance(
    detection_csv=detection_csv,
    df_cat=df_cat,
    output_csv=provenance_csv,
    top_n=cfg.TOP_N_PER_DOMAIN,
)

n_videos = df_prov["video_id"].nunique()
print(f"Provenance CSV : {provenance_csv}")
print(f"  Rows         : {len(df_prov):,}")
print(f"  Unique videos: {n_videos}")
print()
print(df_prov.head(20).to_string(index=False))

In [ ]:
from detection.visualizer import plot_provenance

prov_path = plot_provenance(df_prov, cfg.TOP_N_PER_DOMAIN, dirs["figures"])
print(f"Saved: {prov_path}")

---
## Output structure

```
ExoBaby-results/
├── data/
│   ├── cdi_words.csv
│   └── included_categories_valid129.txt
├── frame_data/
│   └── merged_frame_detections_with_metadata_filtered-0.27.csv
└── analysis/
    ├── results/
    │   └── long_tailed_dist_prop_included_categories_filtered-0.27_valid129.csv
    └── figures/
        ├── fig1a_long_tailed_bar_valid129.png  (.pdf)
        ├── fig1b_loglog_valid129.png  (.pdf)
        └── fig2_per_domain_valid129.png
```